<a href="https://colab.research.google.com/github/MuhammaadArsalanZahid/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

**FlyRank Machine Learning — Week 7 Build+**

This notebook turns the validated ML output into a practical, human-reviewed content action playbook.

The playbook is **decision-support**, not an autonomous production system. Every ranked action requires human review before implementation.


## 1. Load the dataset and create the ranked action queue

This notebook uses the same `content_refresh_anonymized.csv` used in ML-07 through ML-09.

The queue is regenerated locally and exported to `work/outputs/`. The CSV is intentionally not required to be committed to GitHub.


In [18]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT = ROOT / "work" / "outputs"
FIG = ROOT / "work" / "figures"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

candidates = [
    ROOT / "data/raw/content_refresh_anonymized.csv",
    ROOT / "content_refresh_anonymized.csv",
    Path("/content/content_refresh_anonymized.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)

if data_path is None:
    raise FileNotFoundError(
        "Upload content_refresh_anonymized.csv to Colab or place it in data/raw/."
    )

df = pd.read_csv(data_path)

def find_col(frame, names):
    for name in names:
        if name in frame.columns:
            return name
    return None

ctr_col = find_col(df, ["ctr", "CTR", "avg_ctr", "average_ctr"])
position_col = find_col(df, ["avg_position", "position", "average_position"])
impressions_col = find_col(
    df, ["impressions_90d", "impressions", "avg_impressions", "impressions_90_days"]
)
client_col = find_col(df, ["client_hash_id", "client_id", "client"])
content_col = find_col(df, ["content_hash_id", "content_id", "content"])

required = {
    "CTR": ctr_col,
    "position": position_col,
    "impressions": impressions_col,
    "client": client_col
}

missing = [k for k,v in required.items() if v is None]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

for c in [ctr_col, position_col, impressions_col]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=[ctr_col, position_col, impressions_col]).copy()

print("Dataset:", data_path)
print("Rows:", len(df))
print("Columns selected:", required)


Dataset: /content/content_refresh_anonymized.csv
Rows: 30000
Columns selected: {'CTR': 'ctr', 'position': 'avg_position', 'impressions': 'impressions_90d', 'client': 'client_id'}


## 2. Ranked actions + reason codes

The action score combines three current-snapshot signals:
- CTR gap versus the typical CTR for its position bucket;
- search visibility through impressions;
- position.

The reason code explains why an item entered the queue.

No future outcome, label, or post-action result is used.


In [19]:

# Position buckets.
df["_position_bucket"] = pd.cut(
    df[position_col],
    bins=[-np.inf, 3, 10, 20, 50, np.inf],
    labels=["1-3", "4-10", "11-20", "21-50", "51+"],
    include_lowest=True
)

position_ctr = (
    df.groupby("_position_bucket", observed=False)[ctr_col]
      .median()
)

df["_expected_ctr"] = (
    df["_position_bucket"].map(position_ctr).astype(float)
)

df["_ctr_gap"] = (
    df["_expected_ctr"] - df[ctr_col]
).clip(lower=0).fillna(0)

# Visibility is scaled by the 75th percentile to avoid a raw-impressions
# magnitude dominating the whole score.
imp_q75 = df[impressions_col].quantile(.75)
imp_q50 = df[impressions_col].quantile(.50)

df["_visibility_score"] = (
    df[impressions_col] / max(imp_q75, 1)
).clip(0, 2)

df["_position_opportunity"] = np.select(
    [
        df[position_col].between(4, 10, inclusive="both"),
        df[position_col].between(11, 20, inclusive="both"),
        df[position_col] > 20
    ],
    [1.0, 0.8, 0.5],
    default=0.3
)

df["action_score"] = (
    df["_ctr_gap"] *
    df["_visibility_score"] *
    df["_position_opportunity"]
)

# One reason code per ranked row.
df["reason_code"] = np.select(
    [
        (df["_ctr_gap"] > 0) &
        (df[position_col].between(4, 10, inclusive="both")) &
        (df[impressions_col] >= imp_q75),

        (df["_ctr_gap"] > 0) &
        (df[position_col].between(11, 20, inclusive="both")) &
        (df[impressions_col] >= imp_q50),

        (df["_ctr_gap"] > 0) &
        (df[position_col] > 20)
    ],
    [
        "CTR_BELOW_POSITION_EXPECTATION",
        "CTR_GAP_WITH_MEANINGFUL_VISIBILITY",
        "CTR_GAP_LOW_POSITION_PRIORITY"
    ],
    default="REVIEW_ONLY"
)

# Action labels.
df["action"] = np.select(
    [
        df["reason_code"].eq("CTR_BELOW_POSITION_EXPECTATION"),
        df["reason_code"].eq("CTR_GAP_WITH_MEANINGFUL_VISIBILITY"),
        df["reason_code"].eq("CTR_GAP_LOW_POSITION_PRIORITY")
    ],
    [
        "Review title and snippet for CTR opportunity",
        "Review content relevance and SERP snippet alignment",
        "Monitor; investigate only with additional evidence"
    ],
    default="Human review required"
)

queue_cols = [
    c for c in [
        client_col,
        content_col,
        ctr_col,
        position_col,
        impressions_col,
        "_position_bucket",
        "_expected_ctr",
        "_ctr_gap",
        "action_score",
        "reason_code",
        "action"
    ] if c is not None and c in df.columns
]

queue = (
    df[queue_cols]
    .sort_values(
        ["action_score", impressions_col],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue.insert(0, "rank", np.arange(1, len(queue) + 1))

print("Top 10 ranked actions:")
display(queue.head(10))


Top 10 ranked actions:


,rank,client_id,content_id,ctr,avg_position,impressions_90d,_position_bucket,_expected_ctr,_ctr_gap,action_score,reason_code,action
0,1,client_19581e27de,content_c8e9d6ab9013,0.00,9.7,208678,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
1,2,client_7f2253d7e2,content_f986bd514b6e,0.00,6.6,22456,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
2,3,client_4e07408562,content_825a9788af8d,0.00,5.6,16786,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
3,4,client_8527a891e2,content_8ba781dafa55,0.00,9.0,16156,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
4,5,client_4e07408562,content_5d5653c4eb4f,0.00,5.7,15101,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
5,6,client_19581e27de,content_847a841969a2,0.00,7.4,14519,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
6,7,client_f369cb89fc,content_c82bc0c24241,0.00,4.3,13676,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
7,8,client_4e07408562,content_9983d31c53cb,0.00,5.5,7737,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
8,9,client_19581e27de,content_d3aaf7d5f2fc,0.00,8.3,7732,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
9,10,client_f369cb89fc,content_453722754fea,0.01,7.6,140079,4-10,0.16,0.15,0.30,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity


## 3. Archetype → action mapping

These archetypes convert the numeric ranking into a human-readable workflow.

The mapping is intentionally conservative: a model/rule score recommends **what to review**, not an automatic content change.


In [20]:

archetype_map = pd.DataFrame([
    {
        "archetype": "High-visibility CTR gap",
        "signal": "CTR below position-bucket expectation + strong impressions",
        "recommended_action": "Review title, snippet alignment, search intent, and page relevance",
        "human_review": "Required"
    },
    {
        "archetype": "Mid-position CTR opportunity",
        "signal": "Position 11-20 + measurable CTR gap",
        "recommended_action": "Review content depth, intent match, internal linking, and snippet opportunity",
        "human_review": "Required"
    },
    {
        "archetype": "Low-priority CTR gap",
        "signal": "CTR gap but weak position/opportunity",
        "recommended_action": "Monitor and collect additional evidence before editing",
        "human_review": "Required"
    },
    {
        "archetype": "Review-only",
        "signal": "No strong baseline signal",
        "recommended_action": "Do not force an action; investigate manually if business context requires it",
        "human_review": "Required"
    }
])

display(archetype_map)


,archetype,signal,recommended_action,human_review
0,High-visibility CTR gap,CTR below position-bucket expectation + strong...,"Review title, snippet alignment, search intent...",Required
1,Mid-position CTR opportunity,Position 11-20 + measurable CTR gap,"Review content depth, intent match, internal l...",Required
2,Low-priority CTR gap,CTR gap but weak position/opportunity,Monitor and collect additional evidence before...,Required
3,Review-only,No strong baseline signal,Do not force an action; investigate manually i...,Required


## 4. Decay / refresh insight

A refresh decision should not be triggered by age alone.

The current dataset does not, by itself, prove that content decay causes future traffic loss. Therefore this playbook treats staleness/refresh as a **review signal** only when an appropriate freshness field is actually present.


In [21]:

date_like_cols = [
    c for c in df.columns
    if any(token in c.lower() for token in ["date", "updated", "published", "refresh"])
]

print("Potential freshness/date columns available:")
print(date_like_cols if date_like_cols else "None detected.")

if date_like_cols:
    print(
        "\nUse these fields only after checking their semantics and time availability. "
        "Do not interpret correlation with age as causal decay without temporal validation."
    )
else:
    print(
        "\nNo suitable freshness field was detected. Therefore this notebook does "
        "not manufacture a decay metric."
    )


Potential freshness/date columns available:
['days_since_last_update', 'is_initial_refresh_candidate']

Use these fields only after checking their semantics and time availability. Do not interpret correlation with age as causal decay without temporal validation.


## 5. Intended use and limits

### Intended use
- Prioritize pages for human review.
- Give reviewers a transparent reason code.
- Support content-refresh triage.
- Create a repeatable queue for analysis.

### Limits
- The score is directional decision-support.
- It does not establish causality.
- It does not guarantee traffic, ranking, CTR, or business improvement.
- It should not be treated as a production automation system.
- Proxy targets/signals can reflect measurement or data-quality issues.

### What should NOT be automated
- Publishing or deleting content.
- Automatically changing titles/meta descriptions.
- Automatically rewriting factual claims.
- Automatically changing canonical tags, redirects, robots rules, or indexing controls.
- Automatically deciding that a page is low quality.
- Automatically claiming that an edit caused SEO improvement.


## 6. Human review rules

Every ranked action must pass these checks before implementation:

1. Verify search intent and the actual SERP context.
2. Confirm the page is still relevant to the business.
3. Check for brand/legal/compliance constraints.
4. Check recent manual changes or campaigns.
5. Verify that the recommendation is not caused by tracking/data-quality problems.
6. Record the reviewer decision.
7. Measure any implemented change separately rather than treating the model score as proof of impact.


In [22]:

human_review_checklist = pd.DataFrame([
    ["Search intent", "Does the page satisfy the query intent?", "Required"],
    ["SERP context", "Are competitors/SERP features consistent with the recommendation?", "Required"],
    ["Business relevance", "Is the page strategically important?", "Required"],
    ["Data quality", "Could tracking or measurement explain the signal?", "Required"],
    ["Recent changes", "Was the page recently edited or affected by another campaign?", "Required"],
    ["Safety/compliance", "Could an edit create factual, legal, or brand risk?", "Required"],
    ["Measurement plan", "If changed, is there a separate way to evaluate the outcome?", "Required"],
], columns=["Check", "Question", "Status"])

display(human_review_checklist)


,Check,Question,Status
0,Search intent,Does the page satisfy the query intent?,Required
1,SERP context,Are competitors/SERP features consistent with ...,Required
2,Business relevance,Is the page strategically important?,Required
3,Data quality,Could tracking or measurement explain the signal?,Required
4,Recent changes,Was the page recently edited or affected by an...,Required
5,Safety/compliance,"Could an edit create factual, legal, or brand ...",Required
6,Measurement plan,"If changed, is there a separate way to evaluat...",Required


## 7. Monitoring / retrain triggers

The system should be monitored before increasing automation.

### Suggested monitoring triggers
- Feature distributions shift materially from the development data.
- Client mix changes substantially.
- CTR/position/impression distributions change.
- Reviewers repeatedly reject the same reason code.
- Precision@K declines on a later labelled evaluation set.
- A new reliable outcome label becomes available.

### Retraining trigger
Retrain only when there is enough new, trustworthy data and a clearly defined evaluation target. Do not retrain simply because the model is old.


In [23]:

monitoring_triggers = pd.DataFrame([
    ["Feature drift", "Current feature distributions materially differ from training data", "Investigate"],
    ["Client mix drift", "New client population differs from development population", "Investigate / revalidate"],
    ["Reviewer disagreement", "Repeated rejection of a reason code or archetype", "Audit rule"],
    ["Measured performance drop", "Precision@K declines on a later labelled set", "Revalidate / retrain"],
    ["New outcome label", "A reliable future outcome becomes available", "Consider supervised retraining"],
    ["Data-quality issue", "Tracking/schema changes affect signals", "Pause and fix data"],
], columns=["Trigger", "Condition", "Response"])

display(monitoring_triggers)


,Trigger,Condition,Response
0,Feature drift,Current feature distributions materially diffe...,Investigate
1,Client mix drift,New client population differs from development...,Investigate / revalidate
2,Reviewer disagreement,Repeated rejection of a reason code or archetype,Audit rule
3,Measured performance drop,Precision@K declines on a later labelled set,Revalidate / retrain
4,New outcome label,A reliable future outcome becomes available,Consider supervised retraining
5,Data-quality issue,Tracking/schema changes affect signals,Pause and fix data


## 8. Cost / value thinking

The queue should prioritize actions where the expected review value justifies human effort.

A high score is not automatically high business value. Review effort, page importance, traffic opportunity, implementation risk, and expected learning value should be considered together.


In [24]:

cost_value_framework = pd.DataFrame([
    ["High score + high visibility + low review risk", "High", "Prioritize human review"],
    ["High score + unclear business relevance", "Medium", "Review business context first"],
    ["Low score + low visibility", "Low", "Monitor rather than edit"],
    ["Any score + high legal/brand risk", "High risk", "Do not automate; specialist review"],
    ["Any score + suspected data-quality problem", "Unknown", "Fix data before action"],
], columns=["Situation", "Expected value", "Recommended handling"])

display(cost_value_framework)


,Situation,Expected value,Recommended handling
0,High score + high visibility + low review risk,High,Prioritize human review
1,High score + unclear business relevance,Medium,Review business context first
2,Low score + low visibility,Low,Monitor rather than edit
3,Any score + high legal/brand risk,High risk,Do not automate; specialist review
4,Any score + suspected data-quality problem,Unknown,Fix data before action


## 9. Export queue for the research paper

The exact queue used by this notebook is exported to:

`work/outputs/baseline_action_playbook.csv`

This output is regenerated every time the notebook runs and is intentionally kept out of Git when the repository leak-guard requires it.


In [25]:

output_path = OUT / "baseline_action_playbook.csv"

queue.to_csv(output_path, index=False)

# Export a compact methodology/metrics receipt for the paper.
receipt = {
    "rows_in_queue": int(len(queue)),
    "top10_rows": int(min(10, len(queue))),
    "feature_columns": [
    ctr_col,
    position_col,
    impressions_col
],
    "score_definition": "CTR gap × visibility score × position opportunity",
    "reason_code_count": int(queue["reason_code"].nunique()),
    "human_review_required": True,
    "automation_allowed": False,
    "target_type": "current-snapshot decision-support; no future impact claim"
}

receipt_path = OUT / "ml10_playbook_receipt.json"
import json
with open(receipt_path, "w", encoding="utf-8") as f:
    json.dump(receipt, f, indent=2)

print("Queue exported:", output_path)
print("Receipt exported:", receipt_path)
print("Top 10:")
display(queue.head(10))


Queue exported: /content/work/outputs/baseline_action_playbook.csv
Receipt exported: /content/work/outputs/ml10_playbook_receipt.json
Top 10:


,rank,client_id,content_id,ctr,avg_position,impressions_90d,_position_bucket,_expected_ctr,_ctr_gap,action_score,reason_code,action
0,1,client_19581e27de,content_c8e9d6ab9013,0.00,9.7,208678,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
1,2,client_7f2253d7e2,content_f986bd514b6e,0.00,6.6,22456,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
2,3,client_4e07408562,content_825a9788af8d,0.00,5.6,16786,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
3,4,client_8527a891e2,content_8ba781dafa55,0.00,9.0,16156,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
4,5,client_4e07408562,content_5d5653c4eb4f,0.00,5.7,15101,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
5,6,client_19581e27de,content_847a841969a2,0.00,7.4,14519,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
6,7,client_f369cb89fc,content_c82bc0c24241,0.00,4.3,13676,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
7,8,client_4e07408562,content_9983d31c53cb,0.00,5.5,7737,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
8,9,client_19581e27de,content_d3aaf7d5f2fc,0.00,8.3,7732,4-10,0.16,0.16,0.32,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity
9,10,client_f369cb89fc,content_453722754fea,0.01,7.6,140079,4-10,0.16,0.15,0.30,CTR_BELOW_POSITION_EXPECTATION,Review title and snippet for CTR opportunity


## 10. Self-check

The final checks confirm that:
- the ranked queue exists;
- every ranked row has a score, reason code, and action;
- exports were generated;
- human review is required;
- production automation is explicitly disabled;
- the notebook does not claim causal impact.


In [26]:

assert len(queue) > 0
assert "rank" in queue.columns
assert "action_score" in queue.columns
assert "reason_code" in queue.columns
assert "action" in queue.columns

assert output_path.exists()
assert receipt_path.exists()

assert queue["action_score"].notna().all()
assert queue["reason_code"].notna().all()
assert queue["action"].notna().all()

assert receipt["human_review_required"] is True
assert receipt["automation_allowed"] is False

print("PASS: ranked action queue generated.")
print("PASS: score + reason code + action exist.")
print("PASS: queue export exists.")
print("PASS: paper receipt exists.")
print("PASS: human review is required.")
print("PASS: production automation is disabled.")
print("PASS: ML-10 action playbook complete.")


PASS: ranked action queue generated.
PASS: score + reason code + action exist.
PASS: queue export exists.
PASS: paper receipt exists.
PASS: human review is required.
PASS: production automation is disabled.
PASS: ML-10 action playbook complete.
